# total_GVA — extended feature set for TabPFN
- **CAAFE-10** — the LLM-selected engineered features (baseline).
- **Extended** — CAAFE-10 plus a small block of raw scale/skills predictors that
  CAAFE omitted (worker counts, enterprise counts, Level 3 qualifications).

In [4]:
import numpy as np
import pandas as pd
import warnings
from sklearn.model_selection import GroupKFold
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    mean_absolute_percentage_error,
    r2_score,
)
from tabpfn import TabPFNRegressor
from tabpfn.constants import ModelVersion

warnings.filterwarnings("ignore")

FEATURES_PATH = "D:/github/G23-Swindon-Borough-Council\data preprocessing+EDA/features.csv"
ENGINEERED_PATH = "total_gva_engineered_features.csv"
TARGET = "log_total_GVA_2023"

CAAFE_FEATURES = [
    "log_voa_rv_2023",
    "rv_per_working_age",
    "sme_density",
    "qualification_index",
    "firm_size_diversity",
    "rv_per_employee",
    "sme_qual_interaction",
    "employment_quality",
    "modern_sector_leverage",
    "asset_growth_diversity",
]

L3_QUALIFICATIONS = (
    "Level 3 qualifications: 2 or more A levels or VCEs, 4 or more AS levels, "
    "Higher School Certificate, Progression or Advanced Diploma, Welsh "
    "Baccalaureate Advance Diploma, NVQ level 3; Advanced GNVQ, City and Guilds "
    "Advanced Craft, ONC, OND, BTEC National, RSA Advanced Diploma %"
)

SCALE_SKILLS_BLOCK = [
    "total_employees",
    "part_time_employees",
    "total_enterprises_2025_msoa",
    "LU_large_2025_msoa",
    L3_QUALIFICATIONS,
]

In [5]:
raw = pd.read_csv(FEATURES_PATH, low_memory=False).drop_duplicates("LSOA21CD")
engineered = pd.read_csv(ENGINEERED_PATH).drop_duplicates("LSOA21CD")

keys = engineered[["LSOA21CD", "MSOA21CD", TARGET, "is_swindon"] + CAAFE_FEATURES]
data = (
    raw.drop(columns=["MSOA21CD"])
    .merge(keys, on="LSOA21CD", how="inner")
    .dropna(subset=[TARGET, "MSOA21CD"])
    .reset_index(drop=True)
)

feature_sets = {
    "CAAFE-10": CAAFE_FEATURES,
    "Extended": CAAFE_FEATURES + SCALE_SKILLS_BLOCK,
}

print("rows", data.shape[0],
      "| Swindon", int((data["is_swindon"] == "Swindon").sum()),
      "| Others", int((data["is_swindon"] == "Others").sum()))

rows 1121 | Swindon 137 | Others 984


In [6]:
SKEW_THRESHOLD = 1.5
WINSOR_LOWER, WINSOR_UPPER = 0.01, 0.99


def choose_log_columns(train_frame):
    columns = []
    for column in train_frame.columns:
        series = train_frame[column]
        if series.notna().any() and (series.dropna() >= 0).all() and series.skew() > SKEW_THRESHOLD:
            columns.append(column)
    return columns


def preprocess(train_frame, other_frame):
    train_frame = train_frame.copy()
    other_frame = other_frame.copy()
    log_columns = choose_log_columns(train_frame)
    for column in log_columns:
        train_frame[column] = np.log1p(train_frame[column].clip(lower=0))
        other_frame[column] = np.log1p(other_frame[column].clip(lower=0))
    medians = train_frame.median()
    train_frame = train_frame.fillna(medians)
    other_frame = other_frame.fillna(medians)
    lower = train_frame.quantile(WINSOR_LOWER)
    upper = train_frame.quantile(WINSOR_UPPER)
    return (
        train_frame.clip(lower, upper, axis=1).to_numpy(dtype=float),
        other_frame.clip(lower, upper, axis=1).to_numpy(dtype=float),
    )


def score(y_true, y_pred):
    return {
        "R2": r2_score(y_true, y_pred),
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "MAPE": mean_absolute_percentage_error(y_true, y_pred),
    }

## Evaluate both feature sets with TabPFN

In [7]:
y = data[TARGET].to_numpy(dtype=float)
groups = data["MSOA21CD"].to_numpy()
is_swindon = data["is_swindon"].to_numpy()


def tabpfn_groupkfold(feature_names, n_splits=5):
    gkf = GroupKFold(n_splits=n_splits)
    oof = np.empty(len(y))
    for tr, va in gkf.split(data[feature_names], y, groups):
        X_tr, X_va = preprocess(data.loc[tr, feature_names], data.loc[va, feature_names])
        reg = TabPFNRegressor.create_default_for_version(ModelVersion.V3)
        reg.fit(X_tr, y[tr])
        oof[va] = np.asarray(reg.predict(X_va)).reshape(-1)
    return oof


def tabpfn_swindon(feature_names):
    tr = is_swindon == "Others"
    te = is_swindon == "Swindon"
    X_tr, X_te = preprocess(data.loc[tr, feature_names], data.loc[te, feature_names])
    reg = TabPFNRegressor.create_default_for_version(ModelVersion.V3)
    reg.fit(X_tr, y[tr])
    return y[te], np.asarray(reg.predict(X_te)).reshape(-1)


rows = []
for set_name, feature_names in feature_sets.items():
    oof = tabpfn_groupkfold(feature_names)
    rows.append({"feature_set": set_name, "n_features": len(feature_names),
                 "evaluation": "GroupKFold OOF", **score(y, oof)})
    y_te, pred = tabpfn_swindon(feature_names)
    rows.append({"feature_set": set_name, "n_features": len(feature_names),
                 "evaluation": "train=Others, test=Swindon", **score(y_te, pred)})

results = pd.DataFrame(rows)

## Comparison

In [8]:
results.style.format({"R2": "{:.4f}", "MAE": "{:.4f}", "RMSE": "{:.4f}", "MAPE": "{:.2%}"})

,feature_set,n_features,evaluation,R2,MAE,RMSE,MAPE
0,CAAFE-10,10,GroupKFold OOF,0.6845,0.3863,0.6308,10.58%
1,CAAFE-10,10,"train=Others, test=Swindon",0.6987,0.4436,0.7077,12.33%
2,Extended,15,GroupKFold OOF,0.7078,0.3651,0.6070,9.90%
3,Extended,15,"train=Others, test=Swindon",0.7429,0.4105,0.6537,11.38%
